In [1]:
import torch as pt


mols_test = pt.load('./data/mine/test_11499.pt')
print(len(mols_test))
mols_val = pt.load('./data/mine/val_11825.pt')
print(len(mols_val))
mols_all = pt.load('./data/mine/mols_all.pt')
print(len(mols_all))

/tmp/ipykernel_3752833/3602205055.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mols_test = pt.load('./data/mine/test_11499.pt')


11499


/tmp/ipykernel_3752833/3602205055.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mols_val = pt.load('./data/mine/val_11825.pt')


11825


/tmp/ipykernel_3752833/3602205055.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mols_all = pt.load('./data/mine/mols_all.pt')


2253216


In [2]:
from torch.utils.data import DataLoader
from utils.data import SpecDataset, SpecDataset_finetune, collate_fun_emb, collate_fun_finetune


dataset_lib = SpecDataset(mols_all)
loader_lib = DataLoader(dataset_lib, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
dataset_val = SpecDataset(mols_val)
loader_val = DataLoader(dataset_val, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
dataset_test = SpecDataset(mols_test)
loader_test = DataLoader(dataset_test, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
dataset_finetune = SpecDataset_finetune((mols_val, mols_all))

In [4]:
import torch as pt
import torch.optim as optim
from utils.model import Spec2Emb
from tqdm import tqdm
from utils.tools import gen_embeddings, build_idx, evaluate, find_nearest_hit_nhit, save_model
import numpy as np


gpu = 6
model = Spec2Emb().to(gpu)
epochs = 10
batch_size = 32
optimizer = optim.Adam(model.parameters(), lr=0.001)
model_name = 'pure_ft_p0.4'
f = open(model_name+'.txt', 'w') # ft：finetune，base指未更改模型结构

max_metrics = {'expanded': [0, 0], 'insilico': [0, 0], 'expanded_mass': [0, 0], 'insilico_mass': [0, 0]}

for epoch in range(epochs):  
    print(f'==================================Finetune_epoch{epoch+1}======================================')
    f.write('\nFinetune_epoch%d\n' % (epoch+1))
    embeddings_lib = gen_embeddings(model, loader_lib, gpu, power=0.4)
    embeddings_val = gen_embeddings(model, loader_val, gpu, power=0.4)

    I, _ = build_idx(embeddings_lib, embeddings_val, gpu, topk=200) # 内置清缓存
    top1_val, top10_val = evaluate(mols_val, I, mols_all, f, 'Validation')
    vals, hits, nhits = find_nearest_hit_nhit(I, mols_val, mols_all)
    dataset_ft = SpecDataset_finetune(dataset_finetune, mapping=(vals, hits, nhits))
    loader_ft = DataLoader(dataset_ft, batch_size, shuffle=True, num_workers=8, collate_fn=collate_fun_finetune)
    model.train()
    for j in range(5):
        finetune_loss = []
        for i, Data in enumerate(tqdm(loader_ft, unit='batch')):
            Data = [d.to(gpu) for data in Data for d in data]
            optimizer.zero_grad()
            loss = model((Data[:3], Data[3:6], Data[6:9]), mode='finetune', power=0.4)
            finetune_loss.append(loss.item())
            loss.backward()
            optimizer.step()
            if (i+1) %300 ==0:
                loss = np.mean(finetune_loss)
                print(f'Total Loss: {loss}')
                finetune_loss = []

    print(f'===================================Test_epoch{epoch+1}======================================')
    f.write('\n\nTest_epoch%d\n' % (epoch+1))
    embeddings_lib = gen_embeddings(model, loader_lib, gpu, power=0.4) 
    embeddings_test = gen_embeddings(model, loader_test, gpu, power=0.4)
    I_expand, _ = build_idx(embeddings_lib, embeddings_test, gpu, topk=200)
    top1_expand, top10_expand = evaluate(mols_test, I_expand, mols_all, f, 'expanded')
    if top1_expand > max_metrics['expanded'][0] and top10_expand > max_metrics['expanded'][1]:
        max_metrics['expanded'] = [top1_expand, top10_expand]
        save_model(model, model_name, epoch)
    I_insilico, _ = build_idx(embeddings_lib[:2146690], embeddings_test, gpu, topk=200)
    top1_insilico, top10_insilico = evaluate(mols_test, I_insilico, mols_all, f, 'insilico')
    if top1_insilico > max_metrics['insilico'][0] and top10_insilico > max_metrics['insilico'][1]:
        max_metrics['insilico'] = [top1_insilico, top10_insilico]
        save_model(model, model_name, epoch)
    # print(f'\nWith Mass:')
    # f.write('With Mass:\n')
    # embeddings_lib[:, -1] = mass_all
    # embeddings_test[:, -1] = mass_test
    # I_expand, _ = build_idx(embeddings_lib, embeddings_test, gpu, topk=200)
    # top1_expand, top10_expand = evaluate(mols_test, I_expand, mols_all, f, 'expanded')
    # if top1_expand > max_metrics['expanded_mass'][0] and top10_expand > max_metrics['expanded_mass'][1]:
    #     max_metrics['expanded_mass'] = [top1_expand, top10_expand]
    #     save_model(model, model_name, epoch)
    # I_insilico, _ = build_idx(embeddings_lib[:2146690], embeddings_test, gpu, topk=200)
    # top1_insilico, top10_insilico = evaluate(mols_test, I_insilico, mols_all, f, 'insilico')
    # if top1_insilico > max_metrics['insilico_mass'][0] and top10_insilico > max_metrics['insilico_mass'][1]:
    #     max_metrics['insilico_mass'] = [top1_insilico, top10_insilico]
    #     save_model(model, model_name, epoch)
    print(f'================================================================================================')
f.close()

==================================Finetune_epoch1======================================


Searching time:  0:00:01.611630
Validation library
Top1 hit rate: 40.41%
Top10 hit rate: 80.48%


 90%|████████▉ | 316/353 [00:05<00:00, 96.34batch/s]

Total Loss: 1.0081056328614553


 89%|████████▉ | 315/353 [00:07<00:00, 89.26batch/s]

...

 89%|████████▉ | 311/349 [00:07<00:00, 83.66batch/s]

Total Loss: 0.9242295503616333


100%|██████████| 349/349 [00:09<00:00, 37.42batch/s]

===================================Test_epoch10======================================


Searching time:  0:00:01.592337
expanded library
Top1 hit rate: 37.92%
Top10 hit rate: 78.70%
Searching time:  0:00:01.491512
insilico library
Top1 hit rate: 38.22%
Top10 hit rate: 79.09%


In [ ]:
pt.cuda.empty_cache()